# Boston Road Inventory 2022: Acquisition and Initial Cleaning


## Scope and status
Boston (`City = 35`) is the initial pilot, not the final Greater Boston cohort. This notebook loads the saved 2022 layer extract, audits data quality, applies conservative initial cleaning, and exports attributes and source geometries. It is not a final model-ready dataset.

Previously observed: 22,627 input records, 22,592 retained after a basic paths check, and 35 set aside for review. Run the cells to recompute results; these counts are not hard-coded acceptance criteria.

**Run instructions:** place this notebook in `notebooks/` inside the cloned repository. Keep the saved raw JSON in `data/raw/`. Select your Python environment and run top to bottom. Dependencies: pandas, requests, IPython. The optional API download at the end is disabled by default. Original raw files are never overwritten.

## 1. Load the saved raw snapshot

In [1]:
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

# Locate the project root.
cwd = Path.cwd()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / ".git").exists()),
    None
)
if project_root is None:
    raise RuntimeError("Project root not found. Open the project folder in VS Code.")

# Load the raw snapshot saved earlier.
raw_path = (
    project_root / "data" / "raw"
    / "boston_roads_2022_raw_20261016T170046534733Z.json"
)

with raw_path.open("r", encoding="utf-8") as file:
    raw_data = json.load(file)

boston_features = raw_data["features"]
metadata = raw_data["layer_metadata"]

boston_df = pd.DataFrame(
    feature["attributes"] for feature in boston_features
)

assert len(boston_df) == raw_data["record_count"]
print("Loaded records:", len(boston_df))
print("Source file:", raw_path.name)
assert boston_df['OBJECTID'].notna().all()
assert boston_df['OBJECTID'].is_unique
print('Layer:', metadata.get('name'))
print('Filter:', raw_data.get('filter'))
print('Download time (UTC):', raw_data.get('downloaded_at_utc'))


Loaded records: 22627
Source file: boston_roads_2022_raw_20261016T170046534733Z.json
Layer: Road Inventory 2022
Filter: City = 35
Download time (UTC): 2026-10-16T17:00:46.534733+00:00


## 2. Required data and saved field definitions
| Required data | Source |
|---|---|
| Segment geometry | Feature `geometry.paths`, stored separately from CSV |
| Length | `Length`, `Shape__Length` |
| Road class | `F_Class`, `F_F_Class` |
| Lanes | `Num_Lanes`, `Opp_Lanes` |
| Width | `Surface_Wd` |
| Speed | `Speed_Lim`, `Speed`, `Op_Dir_SL` |
| Traffic volume and measurement year | `AADT`, `AADT_Year` |

Field presence does not imply complete values. Units, special values, and the choice among related fields require verification against the [MassDOT dictionary](https://www.mass.gov/doc/road-inventory-data-dictionary/download). Do not assume `Length` and `Shape__Length` have the same units. The layer year is distinct from the AADT measurement year and download time.

In [2]:
required_fields = ['Length', 'Shape__Length', 'F_Class', 'F_F_Class',
                   'Num_Lanes', 'Opp_Lanes', 'Surface_Wd', 'Speed_Lim',
                   'Speed', 'Op_Dir_SL', 'AADT', 'AADT_Year']
missing_fields = sorted(set(required_fields) - set(boston_df.columns))
if missing_fields:
    raise ValueError(f'Required fields absent: {missing_fields}')
fields_df = pd.DataFrame([{'field_name': f['name'], 'alias': f.get('alias'),
                          'data_type': f['type'], 'domain': f.get('domain')}
                         for f in metadata['fields']])
display(fields_df[fields_df['field_name'].isin(required_fields)])


,field_name,alias,data_type,domain
9,F_Class,Functional Class,esriFieldTypeSmallInteger,"{'type': 'codedValue', 'name': 'dStateFunction..."
12,F_F_Class,Federal Functional Class,esriFieldTypeSmallInteger,"{'type': 'codedValue', 'name': 'dFederalFuncti..."
17,Num_Lanes,Number of Travel Lanes,esriFieldTypeInteger,"{'type': 'range', 'name': 'dNumberOfLanes_7', ..."
18,Opp_Lanes,Opposite Number of Travel Lanes,esriFieldTypeInteger,"{'type': 'range', 'name': 'dNumberOfLanes_7', ..."
20,Surface_Wd,Surface Width,esriFieldTypeDouble,None
37,Speed_Lim,Speed Limit,esriFieldTypeSmallInteger,None
38,Op_Dir_SL,Speed Limit Opposing Direction,esriFieldTypeSmallInteger,None
39,Speed,Speed Limit,esriFieldTypeInteger,None
58,AADT,AADT,esriFieldTypeInteger,None
59,AADT_Year,AADT Year,esriFieldTypeSmallInteger,None


## 3. Consolidated data-quality audit
Checks cover all input records before facility eligibility filtering. Repeated IDs are not sufficient evidence of duplicate road geometries. Zero and extreme values are reported without automatically classifying them as errors.

The paths check below only detects whether any path has at least two vertices. It is not a full validity, topology, coordinate, CRS, or spatial duplicate check. Records with unsupported geometry encodings also need review.

In [3]:
import pandas as pd

audit_df = pd.DataFrame(
    feature["attributes"] for feature in boston_features
).replace(r"^\s*$", pd.NA, regex=True)

print(f"Input records: {len(audit_df):,}")

missing_report = pd.DataFrame({
    "missing_count": audit_df.isna().sum(),
    "missing_percent": (audit_df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("\nMissing fields")
display(missing_report[missing_report["missing_count"] > 0])

id_rows = []

for column in ["OBJECTID", "Record_ID", "Rd_Seg_ID", "GlobalID"]:
    values = audit_df[column].dropna()
    id_rows.append({
        "field": column,
        "missing_count": audit_df[column].isna().sum(),
        "unique_non_missing": values.nunique(),
        "rows_with_repeated_id": values.duplicated(keep=False).sum()
    })

id_report = pd.DataFrame(id_rows)

print("\nIdentifier checks")
display(id_report)

numeric_rows = []

for column in [
    "Length", "Shape__Length", "Num_Lanes",
    "Surface_Wd", "Speed_Lim", "Speed", "AADT"
]:
    original = audit_df[column]
    values = pd.to_numeric(original, errors="coerce")

    numeric_rows.append({
        "field": column,
        "missing": original.isna().sum(),
        "non_numeric": (original.notna() & values.isna()).sum(),
        "zero": (values == 0).sum(),
        "negative": (values < 0).sum(),
        "min": values.min(),
        "max": values.max()
    })

numeric_report = pd.DataFrame(numeric_rows)

print("\nNumeric checks")
display(numeric_report)

aadt = pd.to_numeric(audit_df["AADT"], errors="coerce")
years = pd.to_numeric(audit_df["AADT_Year"], errors="coerce")

aadt_report = pd.DataFrame({
    "status": [
        "AADT and year available",
        "AADT available, year missing",
        "AADT missing, year available",
        "Both missing"
    ],
    "record_count": [
        (aadt.notna() & years.notna()).sum(),
        (aadt.notna() & years.isna()).sum(),
        (aadt.isna() & years.notna()).sum(),
        (aadt.isna() & years.isna()).sum()
    ]
})

print("\nAADT completeness")
display(aadt_report)

print("\nAADT measurement years")
display(
    years.value_counts().sort_index()
    .rename_axis("AADT_Year")
    .reset_index(name="record_count")
)
print("Measurement years after 2022:", (years > 2022).sum())

facility_field = next(
    field for field in metadata["fields"]
    if field["name"] == "Facility"
)
codes = (facility_field.get("domain") or {}).get("codedValues", [])
name_map = {item["code"]: item["name"] for item in codes}

facility_report = (
    audit_df.groupby("Facility", dropna=False)
    .size().reset_index(name="record_count")
)
facility_report["facility_name"] = (
    facility_report["Facility"].map(name_map)
)
facility_report["percent"] = (
    facility_report["record_count"] / len(audit_df) * 100
).round(2)

print("\nFacility distribution")
display(facility_report.sort_values("record_count", ascending=False))

geometry_rows = []

for feature in boston_features:
    attributes = feature["attributes"]
    geometry = feature.get("geometry") or {}
    paths = geometry.get("paths") or []

    geometry_rows.append({
        "OBJECTID": attributes["OBJECTID"],
        "geometry_missing_or_empty": not any(
            len(path) >= 2 for path in paths
        ),
        "shape_length_missing": pd.isna(
            attributes.get("Shape__Length")
        )
    })

geometry_report = pd.DataFrame(geometry_rows)

print("\nBasic paths and length completeness")
display(
    pd.crosstab(
        geometry_report["geometry_missing_or_empty"],
        geometry_report["shape_length_missing"],
        rownames=["Fails basic paths check"],
        colnames=["Shape__Length missing"]
    )
)

print("\nAudit complete. No records removed, imputed, or written to the raw file.")

Input records: 22,627

Missing fields


,missing_count,missing_percent
Structure,22627,100.00
Path_Width,22627,100.00
Cr_Slope,22627,100.00
Pd_Sf_Type,22611,99.93
Pd_Fc_Type,22598,99.87
Trk_Netwrk,22400,99.00
Toll_Road,22376,98.89
Speed,22363,98.83
MHS,22146,97.87
StationID,22004,97.25



Identifier checks


,field,missing_count,unique_non_missing,rows_with_repeated_id
0,OBJECTID,0,22627,0
1,Record_ID,37,22590,0
2,Rd_Seg_ID,13699,6660,3925
3,GlobalID,0,22627,0



Numeric checks


...

Measurement years after 2022: 0

Facility distribution


,Facility,record_count,facility_name,percent
0,1.0,18398,Mainline roadway,81.31
10,NaN,3288,NaN,14.53
4,7.0,390,Simple Ramp/ Channelized Turning Lane,1.72
9,12.0,246,Bicycle/ Pedestrian,1.09
1,3.0,96,Tunnel,0.42
3,5.0,74,Rotary,0.33
8,11.0,73,Simple Ramp - Tunnel,0.32
5,8.0,25,Ramp - NB/EB,0.11
2,4.0,21,Doubledeck,0.09
7,10.0,11,Collector - Distributor,0.05



Basic paths and length completeness


Shape__Length missing,False,True
Fails basic paths check,,
False,22592,0
True,0,35



Audit complete. No records removed, imputed, or written to the raw file.


## 4. Initial cleaning
Preserve all original attributes; add a snapshot identifier from `GlobalID`, trim street names, convert blank cleaned names to missing, and add missing-value flags. Separate records failing the basic paths check into a review table. This is a provisional separation, not a conclusion that all retained geometries are valid or all review geometries invalid.

Do not impute AADT, replace special codes, restrict facility types, or remove spatial duplicates at this stage. The source JSON remains unchanged.

In [5]:
from datetime import datetime, timezone
import pandas as pd

# Build a fresh working table from the saved source records.
roads_clean = pd.DataFrame(
    feature["attributes"] for feature in boston_features
)

# Preserve source identifiers and original street names.
roads_clean["segment_id"] = roads_clean["GlobalID"]
roads_clean["street_name_raw"] = roads_clean["St_Name"]

# Trim surrounding whitespace and standardize blank street names.
roads_clean["street_name_clean"] = (
    roads_clean["St_Name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Identify records that fail the basic geometry completeness check.
geometry_issues = {}
for feature in boston_features:
    paths = (feature.get("geometry") or {}).get("paths") or []
    geometry_issues[feature["attributes"]["OBJECTID"]] = not any(
        len(path) >= 2 for path in paths
    )

roads_clean["geometry_missing_or_empty"] = (
    roads_clean["OBJECTID"].map(geometry_issues)
)
assert roads_clean["geometry_missing_or_empty"].notna().all()

# Separate unresolved records without deleting them from the source.
needs_review = roads_clean["geometry_missing_or_empty"]

roads_review = roads_clean.loc[needs_review].copy()
roads_review["exclusion_reason"] = "Failed basic paths check; geometry review required"

roads_clean = roads_clean.loc[~needs_review].copy()

# Preserve missing values and add explicit quality flags.
roads_clean["street_name_missing"] = (
    roads_clean["street_name_clean"].isna()
)
for column, flag in [
    ("AADT", "aadt_missing"),
    ("AADT_Year", "aadt_year_missing"),
    ("Facility", "facility_missing")
]:
    roads_clean[flag] = roads_clean[column].isna()

# Validate record preservation and identifiers.
assert len(roads_clean) + len(roads_review) == len(boston_features)
assert roads_clean["segment_id"].notna().all()
assert roads_clean["segment_id"].is_unique



## 5. Export attributes, source geometries, and audit reports
Each run creates a new timestamped directory under `data/processed/`. Attribute CSVs contain no geometry. Retained geometries and metadata are stored as **Esri JSON, not GeoJSON**. Join attributes to source features using `OBJECTID` within this extract; `segment_id` equals the original `GlobalID`. Do not assume identifiers are stable across different snapshots.

Both dataset directories are ignored by Git. Teammates need the same raw file via an agreed sharing method, or can explicitly use the optional downloader. Pushing this notebook alone does not upload datasets.

In [6]:
# Save this cleaning run separately.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
clean_dir = project_root / "data" / "processed" / f"boston_cleaning_{stamp}"
clean_dir.mkdir(parents=True, exist_ok=False)

roads_clean.to_csv(
    clean_dir / "roads_initial_clean_attributes.csv",
    index=False,
    encoding="utf-8-sig"
)
roads_review.to_csv(
    clean_dir / "roads_needing_geometry_review.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save source geometries for retained records, including source metadata.
retained_ids = set(roads_clean["OBJECTID"])
geometry_output = {
    "source_url": raw_data["source_url"],
    "filter": raw_data["filter"],
    "format": "Esri JSON; not GeoJSON",
    "layer_metadata": metadata,
    "downloaded_at_utc": raw_data.get("downloaded_at_utc"),
    "record_count": len(roads_clean),
    "features": [
        feature for feature in boston_features
        if feature["attributes"]["OBJECTID"] in retained_ids
    ]
}

import json
with (clean_dir / "retained_source_features_esri.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(geometry_output, file, ensure_ascii=False)

print("Input records:", len(boston_features))
print("Retained records:", len(roads_clean))
print("Records set aside for geometry review:", len(roads_review))
print("Saved to:", clean_dir)
print("Special codes and facility eligibility remain unresolved.")
# Save audit results for the complete input, before geometry separation.
for name, report in {
    'missing_values': missing_report,
    'id_checks': id_report,
    'numeric_checks': numeric_report,
    'aadt_completeness': aadt_report,
    'facility_distribution': facility_report,
    'basic_geometry_checks': geometry_report,
}.items():
    report.to_csv(clean_dir / f'{name}.csv', index=(name == 'missing_values'))
fields_df.to_json(clean_dir / 'field_definitions.json', orient='records', indent=2)


Input records: 22627
Retained records: 22592
Records set aside for geometry review: 35
Saved to: c:\Users\wendy\Desktop\Boston University\classes\CS506\Final_Project\cs506-final-project\data\processed\boston_cleaning_20261019T012716618731Z
Special codes and facility eligibility remain unresolved.


## 6. Findings, limitations, and meeting decisions
Previous full-input audit found:
- AADT missing in 4,206 records (18.59%).
- AADT measurement year missing in 14,104 records (62.33%); 9,899 records have AADT but no year.
- Known measurement years range from 2009 to 2021. A 2022 inventory does not imply all traffic counts were measured in 2022.
- Street names missing or blank in 2,800 records (12.37%).
- Facility missing in 3,288 records (14.53%).
- 35 records fail the basic paths check and also lack `Shape__Length`.

These are prior observations; use current audit outputs when reporting a new run.

Pending work: verify dictionary units and special codes (including speed values 0/99); inspect full geometry validity and spatial duplicates; decide eligible facility types and final geographic scope; agree dataset sharing and ownership of crash/image processing. No final eligibility filter has been applied.

Later modeling: fit any missing-AADT imputation on training data only. AADT and segment length approximate traffic exposure. Predicting at least one crash is not a crash rate per traveler.

## Appendix. Optional paginated API acquisition — disabled by default
Run only when intentionally acquiring a new extract. This queries the current service for its 2022 layer; it cannot guarantee byte-for-byte reproduction of the previously downloaded file. Keep the original saved snapshot for reproducibility. After download, set `raw_path` in section 1 to the printed filename and rerun from section 1.

This cell saves layer metadata, query scope, timestamp, record count, response spatial reference, and original Esri features. It checks total count and OBJECTID uniqueness; these checks cannot guarantee an atomic snapshot if the server changes during acquisition.

In [7]:
DOWNLOAD_RAW = False

if DOWNLOAD_RAW:
    import requests
    road_url = ('https://gis.massdot.state.ma.us/arcgis/rest/services/'
                'Roads/RoadInventoryYearEndFiles/FeatureServer/8')
    def get_json(url, params):
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        result = response.json()
        if 'error' in result:
            raise RuntimeError(result['error'])
        return result

    layer = get_json(road_url, {'f': 'json'})
    where = 'City = 35'
    total = get_json(road_url + '/query',
                     {'where': where, 'returnCountOnly': 'true', 'f': 'json'})['count']
    downloaded = []
    spatial_reference = None
    for offset in range(0, total, 1000):
        batch = get_json(road_url + '/query', {
            'where': where, 'outFields': '*', 'returnGeometry': 'true',
            'orderByFields': 'OBJECTID ASC', 'resultOffset': offset,
            'resultRecordCount': 1000, 'f': 'json'})
        items = batch.get('features', [])
        if not items:
            raise RuntimeError('Empty batch before completion.')
        downloaded.extend(items)
        batch_sr = batch.get('spatialReference')
        if spatial_reference is not None and batch_sr != spatial_reference:
            raise RuntimeError('Spatial reference changed across batches.')
        spatial_reference = batch_sr
        print(f'Downloaded {len(downloaded):,} / {total:,}')
    ids = [f['attributes']['OBJECTID'] for f in downloaded]
    assert len(downloaded) == total and len(set(ids)) == total
    now = datetime.now(timezone.utc)
    destination = project_root / 'data' / 'raw'
    destination.mkdir(parents=True, exist_ok=True)
    destination = destination / f"boston_roads_2022_raw_{now.strftime('%Y%m%dT%H%M%S%fZ')}.json"
    payload = {'source_url': road_url, 'filter': where,
               'downloaded_at_utc': now.isoformat(), 'record_count': total,
               'layer_metadata': layer, 'spatialReference': spatial_reference,
               'features': downloaded}
    with destination.open('x', encoding='utf-8') as file:
        json.dump(payload, file, ensure_ascii=False)
    print('Saved new extract:', destination)
else:
    print('Download disabled. The saved snapshot is used.')


Download disabled. The saved snapshot is used.
